# TAD Ethereum — Multi-Year Period Analysis

**Inputs:** per-year results JSON files produced by notebook 3  
**Outputs:** a single analysis results JSON containing, for every run:
- Continuous Wasserstein distance series (cross-year gaps stitched)
- Change-point detection (multivariate, via `ruptures`)
- Period characterisation (mean, variance, skewness, …)
- Anomaly detection per period (S-ESD, IQR, Z-score, rolling σ, isolation forest)

**Only the Configuration cell needs editing.**

## 0. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from period_analysis_functions import (
    load_year_results,
    stitch_all_runs,
    detect_changepoints,
    build_periods,
    characterise_all_periods,
    detect_anomalies_all,
    print_anomaly_summary,
    save_analysis_results,
)

print('Imports OK ✓')

---
## 1. Configuration  ← **edit this cell**

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

# ── Year range (inclusive) ─────────────────────────────────────────────────────────────────
YEAR_START = 2020
YEAR_END   = 2025

# ── Which runs to analyse (must match keys in the year results files) ─────────────────
RUNS = [
    'contract_nonFactory_ALL_750',
    'contract_mediumInput_ALL_750',
    'contract_highInput_ALL_750',
    'simple_txs_ALL_750',
]

# ── Input file pattern ────────────────────────────────────────────────────────────────
RESULTS_FILE_PATTERN = 'run_results_V1_{year}.json'

# ── Change-point settings ───────────────────────────────────────────────────────────
N_YEARS   = YEAR_END - YEAR_START + 1
CP_MIN    = N_YEARS       # minimum number of change points
CP_MAX    = N_YEARS + 3   # maximum number of change points
# ruptures model:
#   'l2'  — mean-shift, O(T log T), FAST — recommended for T > 500
#   'l1'  — median-shift, robust to outliers
#   'rbf' — smooth kernel, more sensitive BUT O(T²) — very slow on T > 500
CP_MODEL  = 'l2'

# ── Anomaly detection settings ──────────────────────────────────────────────────────
SESD_MAX_ANOMALIES_FRAC = 0.10   # max fraction of points that can be anomalies
SESD_ALPHA              = 0.05   # significance level
IQR_K                   = 1.5    # flag if |x - median| > IQR_K * IQR
ZSCORE_THRESH           = 3.0    # flag if |z| > threshold
ROLLING_WINDOW          = 7
ROLLING_K               = 2.5
IFOREST_CONTAMINATION   = 0.05
IFOREST_RANDOM_STATE    = 42

# ── Output file ───────────────────────────────────────────────────────────────────
OUTPUT_FILE = Path(f'analysis_contractBreakdown_Simple_V1_{YEAR_START}_{YEAR_END}.json')

# ════════════════════════════════════════════════════════════════════════════
#  Derived — do not edit
# ════════════════════════════════════════════════════════════════════════════
YEARS = list(range(YEAR_START, YEAR_END + 1))

print(f'Years    : {YEARS}')
print(f'Runs     : {RUNS}')
print(f'CP range : [{CP_MIN}, {CP_MAX}]  model={CP_MODEL}')
print(f'Output   : {OUTPUT_FILE}')
print('Configuration set ✓')

---
## 2. Load results files

In [ ]:
print('Loading results files...')
year_data    = load_year_results(YEARS, RESULTS_FILE_PATTERN, RUNS)
loaded_years = sorted(year_data.keys())
print(f'Loaded {len(loaded_years)} years: {loaded_years}')

---
## 3. Stitch continuous Wasserstein series

In [ ]:
all_series, stitch_log = stitch_all_runs(RUNS, year_data, loaded_years)

---
## 4. Change-point detection

In [ ]:
print(f'Running change-point detection on {len(all_series)} signal(s)...')
cp_dates, n_cp, signal_df = detect_changepoints(
    all_series, CP_MIN, CP_MAX, model=CP_MODEL
)

periods = build_periods(cp_dates, signal_df)

print(f'\nPeriods ({len(periods)} total):')
for i, (s, e) in enumerate(periods):
    print(f'  Period {i+1}: {s.date()} → {e.date()}  ({(e - s).days} days)')

---
## 5. Period characterisation

In [ ]:
import pandas as pd

period_stats = characterise_all_periods(all_series, periods)

summary_rows = []
for run_name in all_series:
    for p in period_stats[run_name]:
        summary_rows.append({'run': run_name, **{k: p[k] for k in
            ['period_index', 'start', 'end', 'n_days', 'mean', 'std', 'skewness', 'trend_slope']}})

summary_df = pd.DataFrame(summary_rows)
print('Period characterisation summary:')
print(summary_df.to_string(index=False))

---
## 6. Anomaly detection

In [ ]:
anomaly_results = detect_anomalies_all(
    all_series, periods,
    sesd_max_anomalies_frac = SESD_MAX_ANOMALIES_FRAC,
    sesd_alpha              = SESD_ALPHA,
    iqr_k                   = IQR_K,
    zscore_thresh           = ZSCORE_THRESH,
    rolling_window          = ROLLING_WINDOW,
    rolling_k               = ROLLING_K,
    iforest_contamination   = IFOREST_CONTAMINATION,
    iforest_random_state    = IFOREST_RANDOM_STATE,
)

print_anomaly_summary(anomaly_results)

---
## 7. Save results

In [ ]:
_cfg = dict(
    YEAR_START=YEAR_START, YEAR_END=YEAR_END, RUNS=RUNS,
    CP_MIN=CP_MIN, CP_MAX=CP_MAX, CP_MODEL=CP_MODEL,
    SESD_MAX_ANOMALIES_FRAC=SESD_MAX_ANOMALIES_FRAC, SESD_ALPHA=SESD_ALPHA,
    IQR_K=IQR_K, ZSCORE_THRESH=ZSCORE_THRESH,
    ROLLING_WINDOW=ROLLING_WINDOW, ROLLING_K=ROLLING_K,
    IFOREST_CONTAMINATION=IFOREST_CONTAMINATION,
)

save_analysis_results(
    OUTPUT_FILE, _cfg, all_series, stitch_log,
    cp_dates, periods, period_stats, anomaly_results,
    year_data, loaded_years,
)
print('\nAll done ✓')